# Statistical Analysis: Maji Ndogo Agricultural Dataset

## Objective

This notebook performs statistical analysis on the cleaned Maji Ndogo dataset.

The goal is to understand:
- Distribution of variables
- Central tendency and spread
- Feature relationships
- Statistical significance
- Regression assumptions

The target variable is:

**Standard_yield**

# Import libraries

In [ ]:
cd ..

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

from src.pipeline import create_final_dataset
from src.exploratory_data_analysis import ExploratoryDataAnalysis

In [ ]:
EDA = ExploratoryDataAnalysis()
df_original = EDA.get_data()
df = df_original.copy()



# Data and Descriptive statistics

In [ ]:
EDA.quality_assessment()

In [ ]:
df.info()

In [ ]:
EDA.summary_statistics()

# Target Variable Analysis

In [ ]:
EDA.plot_numerical_distribution("Standard_yield")


In [ ]:
EDA.single_num_variable_desc_statistics("Standard_yield")

The distribution of Standard_yield is approximately normal with mild right skew, centered at mean=0.534 and median=0.529. The standard deviation of 0.112 indicates moderate variability most farms yield between 0.42 and 0.65, but the full range spans 0.17 to 0.90. There are no extreme outliers. 

# Measure of Skewness and outliers

In [ ]:
numeric_cols =EDA.get_numeric_columns()

In [ ]:
def numeric_feature_distribution(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates the skewness for all numeric columns.
    """
    results = []

    for col in numeric_cols:
        skew_value = df[col].skew()

        try:
            skew_value = float(skew_value)
        except (TypeError, ValueError):
            continue

        if skew_value <= -1:
            skew_interp = "Highly Left-Skewed"
        elif skew_value < -0.5:
            skew_interp = "Moderately Left-Skewed"
        elif skew_value <= 0.5:
            skew_interp = "Approximately Symmetric"
        elif skew_value <= 1:
            skew_interp = "Moderately Right-Skewed"
        else:
            skew_interp = "Highly Right-Skewed"

        results.append({
            "feature": col,
            "Skewness": round(skew_value, 3),
            "Skewness_Interpretation": skew_interp
        })

    return pd.DataFrame(results).sort_values(by="Skewness", ascending=False).reset_index(drop=True)

numeric_feature_distribution(df)

### Skewness Analysis Insight

Most numerical features exhibit approximately symmetric distributions, with skewness values between -0.5 and 0.5. The target variable, `Standard_yield`, is also fairly balanced with only slight positive skewness, which is suitable for regression modeling.

However, `Annual_yield`, `Slope`, `Plot_size`, and `Field_Pollution_Level` show high positive skewness, indicating the presence of long right tails and potential extreme values. These features will be further investigated using visualizations such as box plots to determine whether the skewness is caused by natural variations or potential outliers.


# Correlation Analysis

In [ ]:
def target_correlation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns correlations of numerical features with the target variable.
    """
    results = []

    for col in numeric_cols:
        correlation = df[numeric_cols].corr()
        target_corr = (
            correlation["Standard_yield"]
            .sort_values(ascending=False)
            .to_frame()
        )

    return target_corr


target_correlation(df)

In [ ]:
def correlation_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """
    This method returns the correlations of features predictors to target feature
    """
    correlation = pd.DataFrame(df[numeric_cols].corr())
    correlation["Standard_yield"].sort_values(ascending = False)
    return correlation
correlation_matrix(df)


In [ ]:
def correlation_heatmap(df: pd.DataFrame) -> None:
    corr_matrix = df[numeric_cols].corr()
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
    plt.title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()

correlation_heatmap(df)

### Correlation Analysis Insight

The correlation analysis shows that most features have weak linear relationships with the target variable, `Standard_yield`. The strongest positive correlation with yield is observed in `Annual_yield` (0.221), while `Field_Pollution_Level` shows the strongest negative relationship (-0.286). This suggests that `Standard_yield` is likely influenced by multiple factors rather than a single dominant feature.

The analysis also reveals strong correlations between some predictor variables, such as `Plot_size` and `Annual_yield` (0.949), and between temperature-related variables such as `Elevation` and `Min_temperature_C` (0.956). These strong relationships may indicate potential multicollinearity, which will be investigated further using Variance Inflation Factor (VIF) before model training.


# Statistical Significance (Pearson Test)

In [ ]:
def pearson_test(df: pd.DataFrame):
    """
    This method determines if a a relationship is a statistical significant
    """
    corr, p_value = stats.pearsonr(df["Plot_size"], df["Standard_yield"])
    return corr, p_value

corr, p_value = pearson_test(df)

print("Correlation:", corr)
print("p-value:", p_value)

In [ ]:
def pearson_test(df: pd.DataFrame, feature_cols: list = None, target_col: str = "Standard_yield") -> pd.DataFrame:
    """
    Performs Pearson correlation test between multiple features and target variable.
    """
    if feature_cols is None:
        feature_cols = [col for col in numeric_cols if col != target_col]
    
    results = []
    
    for feature in feature_cols:
        clean_df = df[[feature, target_col]].dropna()

        if len(clean_df) < 3:
            results.append({
                'Feature': feature,
                'Correlation': None,
                'P-Value': None,
                'Significant': None,
                'Interpretation': 'Insufficient data'
            })
            continue

        corr, p_value = stats.pearsonr(clean_df[feature], clean_df[target_col])
        

        significant = p_value < 0.05
        
        # Interpretation
        if significant:
            if abs(corr) > 0.7:
                direction = 'Strong positive' if corr > 0 else 'Strong negative'
            elif abs(corr) > 0.3:
                direction = 'Moderate positive' if corr > 0 else 'Moderate negative'
            else:
                direction = 'Weak positive' if corr > 0 else 'Weak negative'
            interpretation = f"{direction} correlation"
        else:
            interpretation = 'Not significant'
        
        results.append({
            'Feature': feature,
            'Correlation': round(corr, 4),
            'P-Value': round(p_value, 4),
            'Significant': 'Yes' if significant else ' No',
            'Interpretation': interpretation
        })
    
    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values('Correlation', ascending=False, key=abs, na_position='last')
    
    return result_df

results = pearson_test(df, target_col="Standard_yield")
print(results.to_string(index=False))